# Code for Recovering Erased Places in the Panama Canal Zone: OCR, Keyword Search and Attestations

This notebook defines the full pipeline for finding the Canal Zone's erased towns, villages and camps in digitized documents: newspapers, reports, letters and georeferenced maps. It goes from downloaded scans to rows ready to paste into the `attestations` sheet of `canal-zone-relational-model.xlsx`, which is what `export.py` turns into the map.

Every part saves its result as a CSV or Excel checkpoint in `outputs/`, so a part can be re-run, or the notebook restarted, without starting from scratch.

## Setup

### a. Installs (run once)

Tesseract itself is installed with Homebrew in the Terminal, not with pip: `brew install tesseract tesseract-lang` (the second package adds the Spanish and French language files). Ollama must be open while Part 4 runs. The cell below installs the Python packages into the notebook's own environment.

In [ ]:
#pytesseract lets Python call Tesseract, ollama lets Python talk to the local LLM, pymupdf turns PDFs into images,
#pdfplumber reads text that is already inside a PDF, openpyxl reads and writes Excel files
%pip install pytesseract ollama pymupdf pdfplumber openpyxl pandas requests pillow

### b. Imports

In [ ]:
#pandas holds every table and saves the CSV/Excel checkpoints
import pandas as pd
#os builds file paths and checks whether checkpoints already exist, re is for text patterns
import os
import re
import json
import math
import time
import difflib
import hashlib
import subprocess
import unicodedata
#requests downloads files, BytesIO makes downloaded bytes behave like a file
import requests
from io import BytesIO
#PIL opens and prepares images for OCR
from PIL import Image, ImageOps
#pymupdf turns PDF pages into images, pdfplumber pulls out text the PDF already contains
import pymupdf
import pdfplumber
#pytesseract runs the OCR
import pytesseract
#ollama talks to the LLM running locally
import ollama
#openpyxl reads the workbook
import openpyxl

#Big map scans are larger than PIL's default safety limit
Image.MAX_IMAGE_PIXELS = None

### c. Paths and Settings

Every setting the pipeline uses lives in this one cell. The search settings (Part 3) are the criteria the project's harvests used: the ±150-character window, whole-word matching, the context words that pair a name with Panama, and the list of names that are also ordinary words or other places.

In [ ]:
#The notebook sits in ocr_pipeline/, one folder below the workbook
PROJECT = os.path.abspath("..")
WORKBOOK = os.path.join(PROJECT, "canal-zone-relational-model.xlsx")
MASTER_INVENTORY = os.path.join(PROJECT, "data", "georef", "MASTER_INVENTORY.csv")

#Where everything this notebook makes is saved
OUT = "outputs"
PAGES_DIR = os.path.join(OUT, "pages")
os.makedirs(PAGES_DIR, exist_ok=True)
MANIFEST = "manifest.xlsx"

# ── OCR ──────────────────────────────────────────────────────────────────
#English, Spanish and French: the canal's documents are in all three
OCR_LANG = "eng+spa+fra"
PDF_DPI = 300
MIN_TEXT_WIDTH_PX = 2000        #small newspaper scans are enlarged to this width
MAP_TILE_PX = 4000              #big maps are read in tiles this size...
MAP_TILE_OVERLAP_PX = 300       #...overlapping so a label on a tile edge isn't cut
MAP_ROTATIONS = [0, 90, 270]    #Tesseract only reads horizontal text; map names often run sideways
MIN_WORD_CONF = 30              #Tesseract words below this confidence (0-100) are dropped
USE_GOOGLE_VISION = False       #True = send pages to Google Cloud Vision instead of Tesseract
GOOGLE_KEY_ENV = "GOOGLE_VISION_API_KEY"   #the key is read from the Terminal environment, never typed here

# ── Keyword-in-context search ────────────────────────────────────────────
KWIC_WIDTH = 150                #characters kept on each side of a hit
USE_FUZZY = True                #also catch names garbled by OCR ("Pnraiso" for Paraiso)
FUZZY_MIN_RATIO = 0.85
FUZZY_MIN_LEN = 6

#Names that are also ordinary words, rivers, hills, people or other places. In a Canal-only source
#these still need a context word nearby; in a general newspaper every name does.
GUARDED = {
    "Empire", "Summit", "Enterprise", "Darien", "Paraiso", "La Boca", "Red Tank", "Mount Hope",
    "Camacho", "Mindi", "Cruces", "Las Cruces", "Balboa", "Diablo", "Miraflores", "Chagres",
    "Caimito", "Juan Grande", "Frijoles", "Gatuncillo", "Baila Monos", "Casa Blanca", "White House",
    "Whitehouse", "Jamaica Town", "Santa Ana", "Dominica", "Cardenas", "Salsipuedes", "Malambo",
    "Gerig", "Silver City", "Cerro Gordo", "Curundu", "Tivoli", "Dos Hermanas",
}
#Words that show the Canal Zone is being discussed
CONTEXT = re.compile(r"canal|zone|zona|isthm|istmo|panam|chagres|gatun|culebra|commissar|comisariato|"
                     r"i\.?\s?c\.?\s?c|railroad|ferrocarril|colon|col[oó]n|ancon", re.I)
#Words near a hit that suggest it is NOT the settlement. They only fill a "hint" column.
HINTS = {
    "river or stream": re.compile(r"\b(river|rio|r[ií]o|quebrada|creek|brook|stream|rivi[eè]re)\b", re.I),
    "hill": re.compile(r"\b(hill|cerro|mount|mt\.|loma|peak)\b", re.I),
    "person": re.compile(r"\b(mr|mrs|miss|dr|se[nñ]or|sr|don|do[nñ]a|engineer|capt|captain)\.?\s*$", re.I),
    "somewhere else": re.compile(r"colombia|barranquilla|cartagena|cuba|havana|puerto rico|san francisco|"
                                 r"london|long island|morocco|new york|mexico|costa rica|nicaragua", re.I),
    "ship or dredge": re.compile(r"\b(steamer|steamship|dredge|tug|s\.s\.|launch|vapor)\b", re.I),
}
#Phrases that introduce a place name the workbook may not have yet
DISCOVERY = [
    re.compile(r"\b(?:town|village|townsite|settlement|labor camp|camp|hamlet|district)s?\s+(?:of|at)\s+([A-Z][A-Za-z]{3,}(?:\s[A-Z][A-Za-z]+)?)"),
    re.compile(r"\b([A-Z][A-Za-z]{3,}(?:\s[A-Z][A-Za-z]+)?),\s+(?:Canal Zone|C\.\s?Z\.)"),
    re.compile(r"\bstations?\s+(?:of|at)\s+([A-Z][A-Za-z]{3,}(?:\s[A-Z][A-Za-z]+)?)"),
    re.compile(r"\b(?:pueblo|caser[ií]o|villa|aldea|poblado|barrio|campamento)\s+de\s+([A-Z][A-Za-zñáéíóú]{3,}(?:\s[A-Z][A-Za-zñáéíóú]+)?)"),
    re.compile(r"\b(?:village|hameau|chantier)\s+(?:de|d')\s*([A-Z][A-Za-zéèàç]{3,}(?:\s[A-Z][A-Za-zéèàç]+)?)"),
]

# ── LLM ──────────────────────────────────────────────────────────────────
TEXT_MODEL = "mistral"          #any model from `ollama list`
GOLD_SET = os.path.join("gold", "gold_set.csv")
GOLD_DRAFT = os.path.join("gold", "gold_set_draft.csv")

# ── Verdicts and attestations ────────────────────────────────────────────
#The verdict I type -> the attestation_type written to the workbook
VERDICT_TO_TYPE = {
    "confirmed": "Existence / mention",
    "erasure": "Erasure event",
    "name": "Name",
    "geometry": "Geometry (Setting)",
}
VERDICT_SKIP = {"false_positive", "person", "elsewhere", "ship", "unreadable"}
MAP_MERGE_M = 600               #map reads of one place closer than this become one row
LABEL_OFFSET_M = 150            #a label sits beside its feature, so this is added to the uncertainty

### d. Helper for Saving Checkpoints

In [ ]:
#Saves a table as CSV (utf-8-sig so accents open correctly in Excel) and, if asked, also as an Excel file
def save_checkpoint(df, name, excel=False):
    csv_path = os.path.join(OUT, name + ".csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} rows → {csv_path}")
    if excel:
        xlsx_path = os.path.join(OUT, name + ".xlsx")
        df.to_excel(xlsx_path, index=False)
        print(f"Saved {len(df)} rows → {xlsx_path}")


#Loads a checkpoint back, or returns None if it hasn't been made yet
def load_checkpoint(name):
    for ext in (".csv", ".xlsx"):
        path = os.path.join(OUT, name + ext)
        if os.path.exists(path):
            if ext == ".csv":
                return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)
            return pd.read_excel(path, keep_default_na=False)
    return None

## Part 1: Listing and Downloading the Documents

This part of the notebook collects the documents. Nothing is searched here: each document is listed in a manifest, downloaded if needed (never twice), and split into one image per page. Before OCR, the code checks whether a PDF already carries a text layer. Many archive PDFs do (the Internet Archive and the Library of Congress run their own OCR), and that text is kept so it isn't OCR'd again.

### a. The Manifest

`manifest.xlsx` has one row per document. If it doesn't exist yet, the cell creates an empty template and stops. The columns:

| column | meaning |
|---|---|
| `item_id` | my own short unique name, e.g. `EVESTAR_1913-09-14_p5` |
| `source_id` | the workbook source it belongs to, e.g. `S-009`. The source must already have a row in the `sources` sheet |
| `kind` | `text` or `map` |
| `scope` | `general` (a newspaper about everything) or `canal` (a source only about the Zone) |
| `date` | `1913-09-14`, `1913` or `1913-1914` |
| `lang` | `en`, `es` or `fr` |
| `citation` | how the document is cited in the `page / plate` column |
| `url` | where to download it (empty if I already have it) |
| `local_file` | a PDF, image or `.txt` already on this computer |
| `georef_tif` | maps only: the georeferenced GeoTIFF exported from QGIS |
| `rotations` | maps only, optional: lettering angles to try, e.g. `45,315` for a sheet turned diagonally |

In [ ]:
MANIFEST_COLUMNS = ["item_id", "source_id", "kind", "scope", "date", "lang", "citation",
                    "url", "local_file", "georef_tif", "rotations"]

#If there is no manifest yet, make an empty one to fill in, then stop
if not os.path.exists(MANIFEST):
    pd.DataFrame(columns=MANIFEST_COLUMNS).to_excel(MANIFEST, index=False)
    raise SystemExit(f"Created an empty {MANIFEST}. Fill in one row per document, save, and run this cell again.")

#Read everything as text so ids like S-009 and dates stay exactly as typed
manifest = pd.read_excel(MANIFEST, dtype=str, keep_default_na=False)
manifest = manifest[manifest["item_id"].str.strip() != ""]
manifest["kind"] = manifest["kind"].str.strip().str.lower().replace("", "text")
manifest["scope"] = manifest["scope"].str.strip().str.lower().replace("", "general")
print(manifest[["item_id", "source_id", "kind", "scope", "date"]])

### b. Checking the Master Inventory

In [ ]:
#Loads the inventory's source URLs into a dictionary: url -> where the file already is
already_downloaded = {}
if os.path.exists(MASTER_INVENTORY):
    inventory = pd.read_csv(MASTER_INVENTORY, dtype=str, keep_default_na=False)
    for _, r in inventory.iterrows():
        if r["source_url"]:
            already_downloaded[r["source_url"]] = f"{r['folder']}/{r['file']}"

for _, item in manifest.iterrows():
    if item["url"] in already_downloaded:
        print(f"{item['item_id']}: ALREADY DOWNLOADED as {already_downloaded[item['url']]} - put that path in local_file")

### c. Download and Page-Splitting Functions

In [ ]:
#Downloads one file, trying three times before giving up
def download(url, dest):
    for attempt in range(3):
        try:
            resp = requests.get(url, timeout=120, headers={"User-Agent": "CanalZoneGIS/1.0 (dissertation research)"})
            resp.raise_for_status()
            with open(dest, "wb") as f:
                f.write(resp.content)
            return True
        except Exception as e:
            print(f"   attempt {attempt + 1} failed: {e}")
            time.sleep(5 * (attempt + 1))
    return False


#Turns one document into page records: an image path per page, plus any text the PDF already contains
def split_into_pages(item, path):
    records = []
    folder = os.path.join(PAGES_DIR, item["item_id"])
    os.makedirs(folder, exist_ok=True)
    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        doc = pymupdf.open(path)
        with pdfplumber.open(path) as pdf:
            for n, page in enumerate(doc, start=1):
                image_path = os.path.join(folder, f"page_{n:04d}.png")
                if not os.path.exists(image_path):
                    page.get_pixmap(dpi=PDF_DPI).save(image_path)
                #Tries the text layer first, the same way the bills notebook did
                layer = pdf.pages[n - 1].extract_text() or ""
                records.append({"image_path": image_path, "text_layer": layer})
    elif ext == ".txt":
        #Text someone else already OCR'd (an Internet Archive _djvu.txt, a loc.gov full text)
        with open(path, encoding="utf-8", errors="replace") as f:
            records.append({"image_path": "", "text_layer": f.read()})
    else:
        #An image, or a georeferenced map: used where it is, never copied or changed
        records.append({"image_path": path, "text_layer": ""})

    for n, r in enumerate(records, start=1):
        r.update(item_id=item["item_id"], page=n)
    return records

### d. Loop Through the Manifest

In [ ]:
page_records = []
for _, item in manifest.iterrows():
    path = item["local_file"] or item["georef_tif"]

    #No file on this computer yet: download it, unless it was downloaded before
    if not path:
        if not item["url"]:
            print(f"{item['item_id']}: no url and no local_file, skipped")
            continue
        if item["url"] in already_downloaded:
            continue
        folder = os.path.join(OUT, "raw", item["source_id"])
        os.makedirs(folder, exist_ok=True)
        path = os.path.join(folder, item["item_id"] + (os.path.splitext(item["url"].split("?")[0])[1] or ".pdf"))
        if not os.path.exists(path):
            print(f"{item['item_id']}: downloading")
            if not download(item["url"], path):
                continue
            #Pause between downloads so the archive's server isn't hammered
            time.sleep(2)

    pages = split_into_pages(item, path)
    page_records.extend(pages)
    print(f"{item['item_id']}: {len(pages)} page(s)")

df_pages = pd.DataFrame(page_records)
save_checkpoint(df_pages, "01_pages")

## Part 2: Applying OCR

This part turns page images into text. Text documents use Tesseract's automatic layout mode, which finds newspaper columns. Pages that already had a usable text layer are not OCR'd again.

Maps are different: Tesseract's "sparse text" mode looks for words anywhere on the sheet. The map is read in overlapping tiles, and at several angles because map names often run sideways. Each label's position is turned back into the original image's pixels and then into latitude/longitude, using the georeferencing tags QGIS writes into the TIFF.

**What OCR can and can't do here**: Tesseract read the typeset 1920 map well (Taboga, El Vigía and San Juan landed in the right places), but read almost nothing on the hand-lettered, white-on-black 1940 census photostat, even at the right angle. On sheets like that, finding no label means nothing. Must be checked by eye.

### a. Image Preparation Function

In [ ]:
#Prepares an image for OCR: grayscale, stretch the contrast, flip white-on-black photostats, enlarge small scans
def prepare_image(image, kind):
    image = ImageOps.grayscale(image)
    image = ImageOps.autocontrast(image, cutoff=1)
    #A mostly dark image is a photostat (white lines on black); OCR expects dark ink on light paper
    if sum(image.resize((64, 64)).getdata()) / 4096 < 110:
        image = ImageOps.invert(image)
    if kind == "text" and image.width < MIN_TEXT_WIDTH_PX:
        scale = MIN_TEXT_WIDTH_PX / image.width
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    return image

### b. OCR Functions

In [ ]:
#OCR for a text page. Returns the text, the average word confidence and the number of words.
#--psm 3 is Tesseract's automatic layout mode, which handles newspaper columns
def ocr_text_page(image):
    if USE_GOOGLE_VISION:
        annotation = google_vision(image)
        return {"text": annotation.get("text", ""), "confidence": None, "word_count": None}
    data = pytesseract.image_to_data(image, lang=OCR_LANG, config="--psm 3", output_type=pytesseract.Output.DICT)
    confidences = [float(c) for c in data["conf"] if float(c) > 0]
    text = pytesseract.image_to_string(image, lang=OCR_LANG, config="--psm 3")
    return {"text": text,
            "confidence": round(sum(confidences) / len(confidences), 2) if confidences else 0,
            "word_count": len(confidences)}


#OCR words scattered over a map tile. --psm 11 means "sparse text": words anywhere, no reading order.
def ocr_map_words(tile):
    data = pytesseract.image_to_data(tile, lang=OCR_LANG, config="--psm 11", output_type=pytesseract.Output.DICT)
    words = []
    for i, text in enumerate(data["text"]):
        text = (text or "").strip()
        if text and float(data["conf"][i]) >= MIN_WORD_CONF:
            x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
            words.append({"text": text, "conf": float(data["conf"][i]), "x0": x, "y0": y, "x1": x + w, "y1": y + h})
    return words


#Joins words sitting side by side on one baseline into a label ("PEDRO" + "MIGUEL")
def group_into_labels(words):
    labels = []
    for w in sorted(words, key=lambda w: (w["y0"], w["x0"])):
        h = w["y1"] - w["y0"]
        for lab in labels:
            lh = lab["y1"] - lab["y0"]
            same_line = abs((lab["y0"] + lab["y1"]) / 2 - (w["y0"] + w["y1"]) / 2) < 0.5 * max(h, lh)
            close = 0 <= w["x0"] - lab["x1"] < 1.5 * max(h, lh)
            if same_line and close:
                lab["text"] += " " + w["text"]
                lab["confs"].append(w["conf"])
                lab["x1"], lab["y0"], lab["y1"] = w["x1"], min(lab["y0"], w["y0"]), max(lab["y1"], w["y1"])
                break
        else:
            labels.append({**w, "confs": [w["conf"]]})
    for lab in labels:
        confs = lab.pop("confs")
        lab["conf"] = round(sum(confs) / len(confs), 1)
    return labels


#Optional: Google Cloud Vision. Better than Tesseract on maps, faded print and some handwriting; paid per image
#beyond a free monthly allowance. The API key is set in the Terminal with: export GOOGLE_VISION_API_KEY=...
def google_vision(image):
    key = os.environ.get(GOOGLE_KEY_ENV)
    if not key:
        raise SystemExit(f"Set the key in the Terminal first: export {GOOGLE_KEY_ENV}=...")
    import base64
    buffer = BytesIO()
    image.convert("L").save(buffer, format="JPEG", quality=90)
    body = {"requests": [{"image": {"content": base64.b64encode(buffer.getvalue()).decode()},
                          "features": [{"type": "DOCUMENT_TEXT_DETECTION"}],
                          "imageContext": {"languageHints": ["en", "es", "fr"]}}]}
    resp = requests.post("https://vision.googleapis.com/v1/images:annotate", json=body,
                         headers={"X-Goog-Api-Key": key}, timeout=300)
    resp.raise_for_status()
    return resp.json()["responses"][0].get("fullTextAnnotation") or {}

### c. Map Geometry Helpers

Two kinds of geometry. **Rotation:** a label found on the map turned by some angle has to be put back where it is on the unturned map. **Georeferencing:** QGIS saves a north-up GeoTIFF with the size of one pixel (tag 33550) and the map coordinate of one pixel (tag 33922), which is enough to turn any pixel into latitude/longitude without GDAL. The project's georeferenced maps are in Web Mercator (EPSG:3857).

In [ ]:
#Turns an image counter-clockwise by any angle, enlarging the canvas so nothing is cut off
def rotate_image(image, angle):
    if angle % 360 == 0:
        return image
    if angle % 90 == 0:
        return image.transpose({90: Image.Transpose.ROTATE_90, 180: Image.Transpose.ROTATE_180,
                                270: Image.Transpose.ROTATE_270}[angle % 360])
    return image.rotate(angle, expand=True, fillcolor=255, resample=Image.BICUBIC)


#A point on the turned image -> the same point on the original W x H image
def unrotate_point(u, v, angle, W, H):
    t = math.radians(angle)
    W2 = W * abs(math.cos(t)) + H * abs(math.sin(t))
    H2 = W * abs(math.sin(t)) + H * abs(math.cos(t))
    du, dv = u - W2 / 2, v - H2 / 2
    return du * math.cos(t) - dv * math.sin(t) + W / 2, du * math.sin(t) + dv * math.cos(t) + H / 2


#Reads the georeferencing tags from a QGIS GeoTIFF
def geotiff_transform(path):
    with Image.open(path) as im:
        tags = im.tag_v2
        if 33550 not in tags or 33922 not in tags:
            raise ValueError(f"{path} has no georeferencing tags")
        sx, sy = tags[33550][0], tags[33550][1]
        i, j, _, X, Y, _ = tags[33922][:6]
        keys = list(tags.get(34735, ()))
        size = im.size
    epsg = None
    #The GeoKey directory is a list of (key, location, count, value); 3072 = projected CRS, 2048 = geographic CRS
    for k in range(4, len(keys), 4):
        if keys[k] in (3072, 2048):
            epsg = keys[k + 3]
            if keys[k] == 3072:
                break
    if epsg not in (3857, 4326):
        raise ValueError(f"{path}: EPSG:{epsg}, reproject to EPSG:3857 in QGIS first")
    return {"sx": sx, "sy": sy, "i": i, "j": j, "X": X, "Y": Y, "epsg": epsg, "size": size}


#Pixel -> (latitude, longitude)
def pixel_to_latlng(tf, px, py):
    X = tf["X"] + (px - tf["i"]) * tf["sx"]
    Y = tf["Y"] - (py - tf["j"]) * tf["sy"]
    if tf["epsg"] == 4326:
        return Y, X
    lng = math.degrees(X / 6378137.0)
    lat = math.degrees(2 * math.atan(math.exp(Y / 6378137.0)) - math.pi / 2)
    return lat, lng


#How many metres one pixel covers on the ground at this latitude
def metres_per_pixel(tf, lat):
    if tf["epsg"] == 4326:
        return tf["sx"] * 111320 * math.cos(math.radians(lat))
    return tf["sx"] * math.cos(math.radians(lat))


#Distance in metres between two (lat, lng) points
def metres_between(a, b):
    p1, p2 = math.radians(a[0]), math.radians(b[0])
    dp, dl = p2 - p1, math.radians(b[1] - a[1])
    h = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6378137.0 * math.asin(math.sqrt(h))

### d. Loop Through Text Pages Extracting Text

If this cell stops partway, running it again carries on from `02_ocr_text.csv` instead of starting over.

In [ ]:
#Loads earlier progress, so finished pages are skipped
df_ocr = load_checkpoint("02_ocr_text")
done = set() if df_ocr is None else set(zip(df_ocr["item_id"], df_ocr["page"].astype(int)))
ocr_records = [] if df_ocr is None else df_ocr.to_dict("records")

text_items = set(manifest.loc[manifest["kind"] == "text", "item_id"])
todo = df_pages[df_pages["item_id"].isin(text_items)]

for n, (_, p) in enumerate(todo.iterrows(), start=1):
    if (p["item_id"], int(p["page"])) in done:
        continue
    layer = str(p["text_layer"] or "")
    #A text layer with a reasonable number of words is used as it is; otherwise the page is OCR'd
    if len(layer.split()) >= 20 or not p["image_path"]:
        result = {"text": layer, "confidence": None, "word_count": len(layer.split())}
        how = "text layer"
    else:
        result = ocr_text_page(prepare_image(Image.open(p["image_path"]), "text"))
        how = "google vision" if USE_GOOGLE_VISION else "tesseract"
    ocr_records.append({"item_id": p["item_id"], "page": int(p["page"]), "text_from": how, **result})
    #Print statement to track progress
    print(f"{n}/{len(todo)} {p['item_id']} p{p['page']}: {how}, {len(result['text'])} characters")
    #Saves every 10 pages as a checkpoint
    if n % 10 == 0:
        save_checkpoint(pd.DataFrame(ocr_records), "02_ocr_text")

df_ocr = pd.DataFrame(ocr_records, columns=["item_id", "page", "text_from", "text", "confidence", "word_count"])
save_checkpoint(df_ocr, "02_ocr_text")

### e. Loop Through Maps Extracting Labels

In [ ]:
label_records = []
for _, item in manifest[manifest["kind"] == "map"].iterrows():
    path = item["georef_tif"] or item["local_file"]
    image = prepare_image(Image.open(path), "map")
    W, H = image.size
    angles = [int(a) for a in item["rotations"].split(",")] if item["rotations"].strip() else MAP_ROTATIONS
    tf = geotiff_transform(path)
    found = []

    for angle in (angles if not USE_GOOGLE_VISION else [0]):
        turned = rotate_image(image, angle)
        step = MAP_TILE_PX - MAP_TILE_OVERLAP_PX
        #Cuts the (turned) map into overlapping tiles and OCRs each one
        for top in range(0, max(turned.height - MAP_TILE_OVERLAP_PX, 1), step):
            for left in range(0, max(turned.width - MAP_TILE_OVERLAP_PX, 1), step):
                tile = turned.crop((left, top, min(left + MAP_TILE_PX, turned.width), min(top + MAP_TILE_PX, turned.height)))
                for lab in group_into_labels(ocr_map_words(tile)):
                    #Moves the label's corners back onto the unturned map
                    corners = [unrotate_point(left + x, top + y, angle, W, H)
                               for x in (lab["x0"], lab["x1"]) for y in (lab["y0"], lab["y1"])]
                    xs, ys = [c[0] for c in corners], [c[1] for c in corners]
                    found.append({"text": lab["text"], "conf": lab["conf"], "rotation": angle,
                                  "x0": min(xs), "y0": min(ys), "x1": max(xs), "y1": max(ys)})
        print(f"{item['item_id']}: angle {angle} done, {len(found)} labels so far")

    #Map linework comes back as one- or two-letter "words", so labels need at least 3 letters.
    #The same label read twice (tile overlap, two angles) is kept once, with its best confidence.
    kept = []
    for lab in sorted(found, key=lambda l: -l["conf"]):
        if sum(ch.isalpha() for ch in lab["text"]) < 3:
            continue
        size = max(lab["x1"] - lab["x0"], lab["y1"] - lab["y0"])
        cx, cy = (lab["x0"] + lab["x1"]) / 2, (lab["y0"] + lab["y1"]) / 2
        if any(k["text"].lower() == lab["text"].lower() and abs((k["x0"] + k["x1"]) / 2 - cx) < size
               and abs((k["y0"] + k["y1"]) / 2 - cy) < size for k in kept):
            continue
        kept.append(lab)

    #Adds the latitude/longitude of each label's centre and its half-size in metres
    for lab in kept:
        lat, lng = pixel_to_latlng(tf, (lab["x0"] + lab["x1"]) / 2, (lab["y0"] + lab["y1"]) / 2)
        half_px = math.hypot(lab["x1"] - lab["x0"], lab["y1"] - lab["y0"]) / 2
        label_records.append({"item_id": item["item_id"], "page": 1, **lab, "lat": round(lat, 6),
                              "lng": round(lng, 6), "label_halfdiag_m": round(half_px * metres_per_pixel(tf, lat))})
    print(f"{item['item_id']}: {len(kept)} labels kept")

df_labels = pd.DataFrame(label_records, columns=["item_id", "page", "text", "conf", "rotation", "x0", "y0",
                                                 "x1", "y1", "lat", "lng", "label_halfdiag_m"])
save_checkpoint(df_labels, "02_map_labels")

## Part 3: Keyword-in-Context (KWIC) Search

This part searches all the text for every place in the workbook and saves each hit with the text on either side of it (keyword in context). Nothing is decided here: the window is what lets me, and the LLM in Part 4, tell whether "Empire" is the Canal Zone town or the British Empire.

**The search criteria:**

1. **What is searched:** the `preferred_name` and every `variant_names` form of every place that isn't Declined or a Duplicate, plus any phrases in `extra_terms.xlsx` (for places without a proper name, like "police station at Culebra", and for what an archive itself calls a place, e.g. Record Group 185's *Zion Hill Reservoir*). A negative is only as good as the name it was searched under.
2. **Matching:** capital letters and accents are ignored; words split across a line break ("Taber-" / "nilla") are rejoined; only whole words match (`Gatun` does not match inside `Gatuncillo`); spaces, hyphens and full stops are flexible (`Mt. Zion` = `Mt Zion`).
3. **OCR errors (fuzzy matching):** a near-miss scoring 0.85 or more counts, for names of 6+ letters. In multi-word names, short words must match exactly, so "Empire and" is not "Empire Bank". Fuzzy hits are marked and get a Low suggested confidence.
4. **Context rule:** in a `general` source (a newspaper about everything), every name needs a context word (canal, zone, isthmus, Panama, Chagres, Colón…) within the window, which is the "name + Panama" pairing the Chronicling America harvest used. In a `canal` source, only the guarded names need one. Maps are their own context. Hits that fail go to `03_kwic_rejected.csv`; skim it once per source, so the rule isn't hiding real hits.
5. **Window:** 150 characters on each side.
6. **Hints:** nearby words suggesting a river, hill, person, another country or a ship fill a `hint` column. On maps, a label containing numbers or more than four words is flagged as legend or table text.
7. **Discovery:** names after "village of", "X, Canal Zone", "pueblo de", "village de" and similar that aren't in the workbook go to `03_discovery.csv` as leads (one document alone is a citation, not a place).

Spanish-language papers: every page of La Estrella is about Panama, so the context rule filters nothing there. Read Spanish homonyms (Emperador, Balboa, la boca, Cruces, Diablo…) as leads, never as automatic hits.

### a. Loading the Places from the Workbook

In [ ]:
#Reads the places sheet. data_only=True reads the values Excel last calculated, not the formulas.
wb = openpyxl.load_workbook(WORKBOOK, read_only=True, data_only=True)
sheet = wb["places"]
header = [str(c or "").split(" ")[0] for c in next(sheet.iter_rows(min_row=1, max_row=1, values_only=True))]
df_places = pd.DataFrame(list(sheet.iter_rows(min_row=2, values_only=True)), columns=header)
df_places = df_places[df_places["place_id"].notna()]
#Withdrawn registrations are left out, as export.py does
df_places = df_places[~df_places["shortlist_status"].astype(str).str.strip().isin(["Declined", "Duplicate"])]

#Coordinates live on their own sheet (row 1 is an instruction, row 2 the header)
coords = {r[0]: (r[3], r[4]) for r in wb["Coordinates"].iter_rows(min_row=3, values_only=True)
          if r[0] and isinstance(r[3], (int, float)) and isinstance(r[4], (int, float))}
wb.close()


#Workbook dates are free text ("pre-1850?", "c.1913", "1911-1914"): this takes the first and last year
def year_bounds(start, end):
    s = [int(y) for y in re.findall(r"(1[5-9]\d\d|20\d\d)", str(start or ""))]
    e = [int(y) for y in re.findall(r"(1[5-9]\d\d|20\d\d)", str(end or ""))]
    return (min(s) if s else None, max(e) if e else None)


places = {}
for _, r in df_places.iterrows():
    names = [r["preferred_name"]] + str(r["variant_names"] or "").split(";")
    places[r["place_id"]] = {
        "name": r["preferred_name"], "type": r["place_type"],
        "names": sorted({n.strip() for n in names if n and str(n).strip() and str(n) != "None"}),
        "years": year_bounds(r["start_date"], r["end_date"]),
        "latlng": coords.get(r["place_id"]),
    }

#One row per (place, phrase)
terms = [{"place_id": pid, "phrase": n, "guarded": n.lower() in {g.lower() for g in GUARDED}, "from": "workbook"}
         for pid, p in places.items() for n in p["names"]]
if os.path.exists("extra_terms.xlsx"):
    for _, r in pd.read_excel("extra_terms.xlsx", dtype=str, keep_default_na=False).iterrows():
        if r["place_id"] and r["phrase"]:
            terms.append({"place_id": r["place_id"], "phrase": r["phrase"].strip(),
                          "guarded": r.get("guarded", "").strip().upper() == "Y", "from": "extra_terms"})
df_terms = pd.DataFrame(terms).drop_duplicates(["place_id", "phrase"])
print(f"{len(places)} places, {len(df_terms)} search phrases")
save_checkpoint(df_terms, "03_search_terms")

### b. Matching Functions

In [ ]:
#Lower-case and remove accents one character at a time, so the folded text has exactly the same length
#as the original and a position found in one is the same position in the other
def fold(text):
    out = []
    for c in text:
        base = unicodedata.normalize("NFD", c)[0].lower()
        out.append(base if len(base) == 1 else c)
    return "".join(out)


#Rejoins words hyphenated across a line break and tidies whitespace
def clean_ocr_text(text):
    text = re.sub(r"(\w)-\s*\n\s*([a-zñáéíóú])", r"\1\2", str(text or ""))
    return re.sub(r"\s+", " ", text).strip()


#Whole-word pattern, flexible about spaces, hyphens and full stops
def term_regex(phrase):
    parts = [re.escape(tok).replace(r"\.", r"\.?") for tok in fold(phrase).split()]
    return re.compile(r"(?<!\w)" + r"[\s\-]*".join(parts) + r"(?!\w)")


#Near-misses caused by OCR: slides a window of as many words as the name has along the text
def fuzzy_matches(folded, phrase, already):
    target = fold(phrase)
    if len(target) < FUZZY_MIN_LEN:
        return []
    k = len(target.split())
    tokens = list(re.finditer(r"\w+", folded))
    found = []
    for i in range(len(tokens) - k + 1):
        a, b = tokens[i].start(), tokens[i + k - 1].end()
        candidate = " ".join(t.group() for t in tokens[i:i + k])
        if abs(len(candidate) - len(target)) > 2 or any(s < b and a < e for s, e in already):
            continue
        sm = difflib.SequenceMatcher(None, candidate, target)
        if sm.real_quick_ratio() < FUZZY_MIN_RATIO or sm.quick_ratio() < FUZZY_MIN_RATIO:
            continue
        ratio = sm.ratio()
        #Short words must be exact; longer words may carry an OCR slip
        words_ok = all(cw == tw if len(tw) < 5 else difflib.SequenceMatcher(None, cw, tw).ratio() >= 0.75
                       for cw, tw in zip(candidate.split(), target.split()))
        if words_ok and FUZZY_MIN_RATIO <= ratio < 1.0:
            found.append((a, b, round(ratio, 3)))
    return found


#Fills the hint column from the words around a hit
def hints_for(left, right):
    found = []
    for label, pattern in HINTS.items():
        if label == "person":
            if pattern.search(left[-40:]):
                found.append(label)
        elif pattern.search(left[-60:] + " " + right[:60]):
            found.append(label)
    return "; ".join(found)


#A stable id for each hit, so my verdicts survive re-running the search
def make_hit_id(*parts):
    return "H-" + hashlib.sha1("|".join(str(p) for p in parts).encode()).hexdigest()[:10]

### c. Loop Through the Text and Map Labels Searching

In [ ]:
regexes = {phrase: term_regex(phrase) for phrase in df_terms["phrase"]}
known_names = {fold(p) for p in df_terms["phrase"]}
items = manifest.set_index("item_id").to_dict("index")

#Builds the list of things to search: each text page, and each map label on its own
units = []
for _, r in df_ocr.iterrows():
    units.append({"item_id": r["item_id"], "page": int(r["page"]), "text": clean_ocr_text(r["text"]), "label": None})
for _, r in df_labels.iterrows():
    units.append({"item_id": r["item_id"], "page": int(r["page"]), "text": r["text"], "label": r.to_dict()})

term_list = df_terms.to_dict("records")
hits, rejected, discovery = [], [], []
log_counts = {}
for u in units:
    item = items[u["item_id"]]
    text, folded = u["text"], fold(u["text"])
    for t in term_list:
        matches = [(m.start(), m.end(), 1.0) for m in regexes[t["phrase"]].finditer(folded)]
        if USE_FUZZY:
            matches += fuzzy_matches(folded, t["phrase"], [(a, b) for a, b, _ in matches])
        for a, b, similarity in matches:
            left, right = text[max(0, a - KWIC_WIDTH):a], text[b:b + KWIC_WIDTH]
            #The context rule (see the notes above this part)
            needs_context = u["label"] is None and (item["scope"] == "general" or t["guarded"])
            context_ok = (not needs_context) or bool(CONTEXT.search(left + right))
            row = {
                "hit_id": make_hit_id(u["item_id"], u["page"], t["place_id"], a, u["label"] and u["label"]["x0"]),
                "source_id": item["source_id"], "item_id": u["item_id"], "kind": item["kind"],
                "date": item["date"], "lang": item["lang"] or "en", "page": u["page"],
                "page_ref": f"{item['citation'] or u['item_id']}, p. {u['page']}",
                "place_id": t["place_id"], "place_name": places.get(t["place_id"], {}).get("name", "?"),
                "phrase": t["phrase"], "matched_text": text[a:b],
                "match_type": "exact" if similarity == 1.0 else "fuzzy", "similarity": similarity,
                "guarded": t["guarded"], "context_ok": context_ok,
                "hint": hints_for(left, right) if u["label"] is None else
                        ("legend or table text? not a location" if re.search(r"\d", text) or len(text.split()) > 4 else ""),
                "left": " ".join(left.split()), "hit": text[a:b], "right": " ".join(right.split()),
                "lat": u["label"]["lat"] if u["label"] else "", "lng": u["label"]["lng"] if u["label"] else "",
                "label_halfdiag_m": u["label"]["label_halfdiag_m"] if u["label"] else "",
                "suggested_confidence": "Medium" if similarity == 1.0 else "Low",
            }
            (hits if context_ok else rejected).append(row)
            key = (item["source_id"], u["item_id"], t["place_id"], t["phrase"])
            log_counts.setdefault(key, {"exact_hits": 0, "fuzzy_hits": 0, "context_rejected": 0})
            if not context_ok:
                log_counts[key]["context_rejected"] += 1
            elif similarity == 1.0:
                log_counts[key]["exact_hits"] += 1
            else:
                log_counts[key]["fuzzy_hits"] += 1

    #Discovery patterns (text only)
    if u["label"] is None:
        for pattern in DISCOVERY:
            for m in pattern.finditer(text):
                if fold(m.group(1)) not in known_names:
                    discovery.append({"candidate": m.group(1), "source_id": item["source_id"], "item_id": u["item_id"],
                                      "date": item["date"], "page": u["page"],
                                      "context": " ".join(text[max(0, m.start() - 100):m.end() + 100].split())})

#The search log records EVERY phrase searched in every item, including zero hits: it is the proof Part 6 needs
log_rows = []
for item_id, item in items.items():
    n_units = sum(1 for u in units if u["item_id"] == item_id)
    for t in term_list:
        c = log_counts.get((item["source_id"], item_id, t["place_id"], t["phrase"]),
                           {"exact_hits": 0, "fuzzy_hits": 0, "context_rejected": 0})
        log_rows.append({"source_id": item["source_id"], "item_id": item_id, "kind": item["kind"],
                         "units_searched": n_units, "place_id": t["place_id"], "phrase": t["phrase"], **c})

df_hits = pd.DataFrame(hits)
save_checkpoint(df_hits, "03_kwic_hits")
if df_hits.empty:
    print("No hits: check the manifest, the OCR text and the search terms before going on.")
save_checkpoint(pd.DataFrame(rejected), "03_kwic_rejected")
save_checkpoint(pd.DataFrame(log_rows), "03_search_log")
save_checkpoint(pd.DataFrame(discovery), "03_discovery", excel=True)

## Part 4: First Reading by a Local LLM (Ollama)

This part uses Mistral running locally in Ollama to take a first pass at every hit, so reading thousands of hits becomes checking suggestions. The model only suggests: its answers go into `llm_` columns, and nothing is imported unless I type the verdict myself in Part 5.

Safeguards built in:
- **Temperature 0 and a fixed seed**, so the same hit always gets the same answer.
- **A fixed answer format** (a JSON schema Ollama enforces), so answers never need parsing by hand.
- **A checked quote:** the model must quote the words around the name that decided its verdict. The code checks the quote really is in the window and is more than the name itself; otherwise the suggestion becomes `unsure`.
- **Only the window:** the prompt forbids using outside knowledge of canal history to add facts or dates.
- **Everything logged:** every prompt and raw answer, with the model name and prompt version, goes to `outputs/llm_log.jsonl` for the methods chapter.


### a. Configuration of Ollama

In [ ]:
#The answer the model must give, as a JSON schema that Ollama enforces
TEXT_VERDICTS = ["confirmed", "erasure", "name", "false_positive", "person", "elsewhere", "ship", "unsure"]
MAP_VERDICTS = ["geometry", "false_positive", "unsure"]
ACCEPT = {"confirmed", "erasure", "name", "geometry"}
REJECT = {"false_positive", "person", "elsewhere", "ship"}
PROMPT_VERSION = "2026-09-17b"


def answer_schema(verdicts):
    return {"type": "object",
            "properties": {"verdict": {"type": "string", "enum": verdicts},
                           "evidence_quote": {"type": "string"},
                           "ocr_reading": {"type": "string"},
                           "reason": {"type": "string"},
                           "needs_full_page": {"type": "boolean"}},
            "required": ["verdict", "evidence_quote", "ocr_reading", "reason", "needs_full_page"]}


#Sends one prompt to the local model and returns its answer as a dictionary
def ask_llm(system_prompt, user_prompt, verdicts, model=TEXT_MODEL):
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
            format=answer_schema(verdicts),
            options={"temperature": 0, "seed": 42, "num_ctx": 4096},
        )
        raw = response["message"]["content"]
        return raw, json.loads(raw)
    except Exception as e:
        return f"[ERROR] {e}", {}


#Checks Ollama is running and the model answers
print(ollama.chat(model=TEXT_MODEL, messages=[{"role": "user", "content": "Say 'ready' and nothing else."}])["message"]["content"])

### b. Prompts

In [ ]:
#The instructions for text hits. Each verdict is defined, and rule 1 keeps the model to the window.
SYSTEM_TEXT = """You help a historian check search hits in digitized historical documents about the \
Panama Canal Zone. The documents were read by OCR, so expect misspellings, broken words and stray characters.

You are given ONE place from the historian's gazetteer and ONE short text window in which a search found \
that place's name. Decide what the name refers to IN THIS WINDOW.

Verdicts:
- confirmed: the words refer to this settlement or site (it existed, people lived or worked there, \
something happened there, it is listed among places).
- erasure: the words refer to this settlement AND record its removal, demolition, abandonment, \
depopulation, condemnation, relocation of residents, or flooding (including "destined to disappear", \
"will be covered by the lake").
- name: the words use the name for the river, hill, reservoir or land tract at the same locality, or use \
the name after the settlement had ended, without saying the settlement exists.
- person: the name belongs to a person.
- ship: the name belongs to a ship, dredge, tug or launch.
- elsewhere: a different place with the same or a similar name (another country, another city, a building \
such as the White House in Washington).
- false_positive: anything else that is not this place (an ordinary word, a plant or wood, a street or \
avenue named after it, a garbled OCR accident).
- unsure: the window is too short or too garbled to decide.

Rules:
1. Use ONLY the window. Do not use your own knowledge of history to add facts, dates or places that the \
window does not state.
2. evidence_quote must be copied EXACTLY, character for character, from the window, without the [[ ]] \
marks, and must be the words AROUND the name that decide the verdict (5 to 25 words). The name alone is \
not evidence.
3. If the decisive words are not in the window, answer unsure and set needs_full_page true.
4. ocr_reading: how the matched words would read without OCR errors (same as the text if clean).
5. reason: one plain sentence."""

#The instructions for map labels
SYSTEM_MAP = """You help a historian check text read by OCR from a historical map of the Panama Canal Zone. \
You get one OCR label and the gazetteer place its words matched.

Verdicts:
- geometry: the label is a place label on the map face naming this place (it marks where the place is).
- false_positive: the label is not a place label for this place: legend or table text, a title, a \
statistic ("Pedro Miguel, 376 feet"), a lock, lake or river named after the place, or OCR noise.
- unsure: cannot tell.

evidence_quote must be copied exactly from the label. Use only the label."""


#Fills in the details of one hit: the place as the workbook records it, the document, and the window
def build_user_prompt(case):
    p = places.get(case["place_id"], {})
    first, last = p.get("years", (None, None))
    other_names = ", ".join(p.get("names", [])) or "none"
    if case["kind"] == "map":
        return (f"PLACE: {p.get('name', case['place_id'])} ({case['place_id']}), a {p.get('type') or 'place'}; "
                f"other names: {other_names}.\nMAP: {case['source']}, dated {case['date'] or '?'}.\n"
                f"OCR LABEL: {case['left']} {case['hit']} {case['right']}\nMATCHED WORDS: {case['hit']}")
    return (f"PLACE: {p.get('name', case['place_id'])} ({case['place_id']}), recorded as a {p.get('type') or 'place'}, "
            f"existing {first or '?'} to {last or 'still existing / unknown'}. Other names: {other_names}.\n"
            f"DOCUMENT: {case['source']}, dated {case['date'] or '?'}, language {case['lang'] or 'en'}.\n"
            f"SEARCHED FOR: {case['phrase']}\n"
            f"WINDOW (the match is between [[ and ]]):\n...{case['left']} [[{case['hit']}]] {case['right']}...")

### c. Helper Function to Ask and Check

In [ ]:
#Squashes text for comparison: folded, single spaces
def squash(text):
    return re.sub(r"\s+", " ", fold(str(text or ""))).strip()


#Asks the model about one hit, checks its quote, logs everything, and returns the suggestion
def review_hit(case, model=TEXT_MODEL):
    is_map = case["kind"] == "map"
    user_prompt = build_user_prompt(case)
    start = time.time()
    raw, answer = ask_llm(SYSTEM_MAP if is_map else SYSTEM_TEXT, user_prompt,
                          MAP_VERDICTS if is_map else TEXT_VERDICTS, model=model)
    result = {"llm_verdict": answer.get("verdict", "unsure"), "llm_quote": answer.get("evidence_quote", ""),
              "llm_ocr_reading": answer.get("ocr_reading", ""), "llm_reason": answer.get("reason", ""),
              "llm_needs_full_page": bool(answer.get("needs_full_page", False)), "llm_flag": "",
              "llm_model": model, "llm_seconds": round(time.time() - start, 1)}

    #The checks: a real answer, a quote that is in the window, and a quote that is more than the name
    window = squash(f"{case['left']} {case['hit']} {case['right']}")
    quote = squash(result["llm_quote"].replace("[[", "").replace("]]", "")).strip(" .,;:'\"")
    other_words = [w for w in re.findall(r"\w{3,}", quote) if w not in squash(case["hit"]).split()]
    said = answer.get("verdict")
    if not answer:
        result.update(llm_verdict="unsure", llm_flag="no valid answer from the model")
    elif not quote:
        result.update(llm_verdict="unsure", llm_flag=f"no quote given (model said {said})")
    elif quote not in window:
        result.update(llm_verdict="unsure", llm_flag=f"quote not found in window (model said {said})")
    elif not is_map and not other_words:
        result.update(llm_verdict="unsure", llm_flag=f"quote is only the name, no evidence (model said {said})")

    #Logs the prompt and raw answer for the record
    with open(os.path.join(OUT, "llm_log.jsonl"), "a", encoding="utf-8") as log:
        log.write(json.dumps({"time": time.strftime("%Y-%m-%dT%H:%M:%S"), "model": model,
                              "prompt_version": PROMPT_VERSION, "case": case.get("hit_id") or case.get("case_id"),
                              "prompt": user_prompt, "raw": raw, "result": result}, ensure_ascii=False) + "\n")
    return result


#Groups verdicts into accept / reject / unsure, for scoring
def family(verdict):
    return "accept" if verdict in ACCEPT else "reject" if verdict in REJECT else "unsure"

### d. Loop Through the Hits and Ask the Model

The loop saves every 20 hits and skips hits already answered, so if it stops it carries on from `04_kwic_hits_llm.csv`.

In [ ]:
input_path_name, output_path_name = "03_kwic_hits", "04_kwic_hits_llm"

#Load prior progress if it exists, otherwise start from the search results
df = load_checkpoint(output_path_name)
if df is None:
    df = load_checkpoint(input_path_name)
    print(f"Starting new from {input_path_name}")
else:
    print(f"Continuing from {output_path_name}")

LLM_COLUMNS = ["llm_verdict", "llm_quote", "llm_ocr_reading", "llm_reason", "llm_needs_full_page",
               "llm_flag", "llm_model", "llm_seconds"]
for col in LLM_COLUMNS:
    if col not in df.columns:
        df[col] = ""

for n, (i, row) in enumerate(df.iterrows(), start=1):
    #Skip hits already answered, important when restarting
    if str(row["llm_verdict"]).strip():
        continue
    case = {**row.to_dict(), "source": items[row["item_id"]]["citation"] or row["item_id"]}
    result = review_hit(case)
    for col in LLM_COLUMNS:
        df.at[i, col] = result[col]
    print(f"{n}/{len(df)} {row['place_id']} {row['hit']!r}: {result['llm_verdict']} ({result['llm_seconds']} s)"
          + (f" [{result['llm_flag']}]" if result["llm_flag"] else ""))
    if n % 20 == 0:
        save_checkpoint(df, output_path_name)

df_llm = df
save_checkpoint(df_llm, output_path_name)

### e. Building the Reading File

The reading file sorts the hits by how much attention they need: first the model's rejections (a wrongly rejected mention is the dangerous error), then unsure answers, failed quote checks and "needs full page", and last the model's acceptances.

In [ ]:
def reading_priority(row):
    if row["llm_verdict"] in REJECT:
        return 1
    if row["llm_verdict"] == "unsure" or row["llm_flag"] or str(row["llm_needs_full_page"]) == "True":
        return 2
    return 3


df_read = df_llm.copy()
df_read["read_order"] = df_read.apply(reading_priority, axis=1)
df_read["read_group"] = df_read["read_order"].map({1: "1 model rejected - read in full",
                                                   2: "2 unsure / flagged - read in full",
                                                   3: "3 model accepted - check the window"})
#Empty columns for me to fill
df_read["verdict"] = ""
df_read["confidence"] = ""
df_read["verdict_note"] = ""
first_columns = ["read_group", "verdict", "confidence", "verdict_note", "place_id", "place_name", "left", "hit", "right",
                 "llm_verdict", "llm_reason", "llm_quote", "llm_flag", "hint", "match_type", "date", "page_ref"]
df_read = df_read.sort_values(["read_order", "place_id", "date"])
df_read = df_read[first_columns + [c for c in df_read.columns if c not in first_columns + ["read_order"]]]

#If I already filled in verdicts in an earlier round, they are carried over by hit_id
filled = load_checkpoint("05_verdicts_FILLED")
if filled is not None:
    earlier = filled.set_index("hit_id")[["verdict", "confidence", "verdict_note"]]
    for col in ["verdict", "confidence", "verdict_note"]:
        df_read[col] = df_read["hit_id"].map(earlier[col]).fillna("")

df_read.to_excel(os.path.join(OUT, "05_verdicts_TO_FILL.xlsx"), index=False)
print(f"Saved {len(df_read)} hits → outputs/05_verdicts_TO_FILL.xlsx")
print(df_read["read_group"].value_counts())

## Part 5: My Verdicts

Open outputs, read every hit, type a verdict in the `verdict` column, and save it as **`outputs/05_verdicts_FILLED.xlsx`**. `confidence` (High / Medium / Low) and `verdict_note` are optional. If `confidence` is empty, the suggested confidence is used.

| verdict | when | becomes |
|---|---|---|
| `confirmed` | the passage refers to the settlement | Existence / mention |
| `erasure` | it records removal, demolition, flooding, condemnation | Erasure event |
| `name` | the name outlives the place, or names the river/hill at the site | Name |
| `geometry` | maps only: the label marks the place's location | Geometry (Setting) |
| `false_positive`, `person`, `elsewhere`, `ship`, `unreadable` | not this place | not imported; named in the negative's note |

In [ ]:
path = os.path.join(OUT, "05_verdicts_FILLED.xlsx")
if not os.path.exists(path):
    raise SystemExit("Fill in outputs/05_verdicts_TO_FILL.xlsx and save it as outputs/05_verdicts_FILLED.xlsx first.")
df_verdicts = pd.read_excel(path, dtype=str, keep_default_na=False)
df_verdicts["verdict"] = df_verdicts["verdict"].str.strip().str.lower()
df_verdicts["confidence"] = df_verdicts["confidence"].str.strip().str.title()

#Checks every verdict and confidence is one of the allowed words
allowed = set(VERDICT_TO_TYPE) | VERDICT_SKIP | {""}
bad_verdicts = df_verdicts[~df_verdicts["verdict"].isin(allowed)]
bad_confidence = df_verdicts[~df_verdicts["confidence"].isin(["", "High", "Medium", "Low"])]
if len(bad_verdicts) or len(bad_confidence):
    print(bad_verdicts[["hit_id", "place_id", "hit", "verdict"]])
    print(bad_confidence[["hit_id", "place_id", "hit", "confidence"]])
    raise SystemExit("Fix the values above, save, and run again.")

unread = (df_verdicts["verdict"] == "").sum()
print(f"{len(df_verdicts) - unread} hits have a verdict, {unread} still unread")

#How often my verdict and the model's point the same way: useful for the methods chapter
read = df_verdicts[df_verdicts["verdict"] != ""]
agreement = pd.crosstab(read["verdict"], read["llm_verdict"], margins=True)
save_checkpoint(df_verdicts, "05_verdicts_checked")
agreement.to_csv(os.path.join(OUT, "05_my_verdicts_vs_llm.csv"), encoding="utf-8-sig")
agreement

## Part 6: Negative Sweep

The thesis treats absence as data, so a negative needs **expected + searched + not found**. For every source and every place, this part decides:

- **Present:** at least one hit I confirmed.
- **Candidate negative:** expected here, every name form searched in every item (`03_search_log.csv` is the proof), and no confirmed hit.
- **Blocked: unread hits:** a hit for this place has no verdict yet.
- **Blocked: walk unfinished:** some items of the source have no OCR yet. Positives can be imported before the walk ends; negatives cannot.
- **Not expected:** outside the place's lifespan, or outside the map sheet.
- **Cannot judge:** a map can't be checked for a place with no coordinate.

**Expected** means the place's lifespan overlaps the source's date and, for a map, its coordinate is inside the sheet.

### a. Classifying Every Source and Place

In [ ]:
df_log = load_checkpoint("03_search_log")
ocr_done = set(df_ocr["item_id"]) | set(df_labels["item_id"])
absence_rows = []

for source_id, group in manifest.groupby("source_id"):
    #The source's years, from its items' dates
    years = [y for d in group["date"] for y in year_bounds(d, d) if y]
    first_year, last_year = (min(years), max(years)) if years else (None, None)
    walk_complete = all(i in ocr_done for i in group["item_id"])
    is_map = (group["kind"] == "map").all()

    #For maps: the extent of each sheet in latitude/longitude
    bounds = []
    if is_map:
        for path in group["georef_tif"]:
            tf = geotiff_transform(path)
            w, h = tf["size"]
            corners = [pixel_to_latlng(tf, x, y) for x in (0, w) for y in (0, h)]
            bounds.append((min(c[0] for c in corners), max(c[0] for c in corners),
                           min(c[1] for c in corners), max(c[1] for c in corners)))

    for place_id, p in places.items():
        start, end = p["years"]
        phrases = sorted(df_log.loc[(df_log["source_id"] == source_id) & (df_log["place_id"] == place_id), "phrase"].unique())
        row = {"source_id": source_id, "place_id": place_id, "place_name": p["name"],
               "source_years": f"{first_year}-{last_year}", "items": len(group),
               "phrases_searched": "; ".join(phrases), "status": "", "reason": "",
               "approve": "", "confidence": "", "note": ""}
        mine = df_verdicts[(df_verdicts["source_id"] == source_id) & (df_verdicts["place_id"] == place_id)]
        confirmed = mine[mine["verdict"].isin(VERDICT_TO_TYPE)]
        ruled_out = mine[mine["verdict"].isin(VERDICT_SKIP)]
        unread = mine[mine["verdict"] == ""]

        if first_year and ((start and start > last_year) or (end and end < first_year)):
            row.update(status="Not expected", reason=f"lifespan {start}-{end} is outside the source's dates")
        elif is_map and not p["latlng"]:
            row.update(status="Cannot judge", reason="place has no coordinate: check the sheet by eye")
        elif is_map and not any(a <= p["latlng"][0] <= b and c <= p["latlng"][1] <= d for a, b, c, d in bounds):
            row.update(status="Not expected", reason="coordinate is outside every sheet")
        elif len(confirmed):
            row.update(status="Present", reason=f"{len(confirmed)} confirmed hit(s)")
        elif len(unread):
            row.update(status="Blocked: unread hits", reason=f"{len(unread)} hit(s) have no verdict")
        elif not walk_complete:
            row.update(status="Blocked: walk unfinished", reason="some items have no OCR yet")
        else:
            where = (f"expected near {p['latlng'][0]:.5f}, {p['latlng'][1]:.5f}; no OCR label for any name form"
                     if is_map else "no confirmed mention in the OCR text")
            ruled = (f" {len(ruled_out)} hit(s) read and ruled not this place: "
                     + "; ".join(f"{r.hit} ({r.verdict})" for r in ruled_out.itertuples()) + ".") if len(ruled_out) else ""
            row.update(status="Candidate negative", reason="expected, searched, not found",
                       note=f"Searched OCR of {len(group)} item(s) for: {row['phrases_searched']}; {where}.{ruled}")
        absence_rows.append(row)

df_absence = pd.DataFrame(absence_rows)
df_absence.to_excel(os.path.join(OUT, "06_absence_candidates.xlsx"), index=False)
print("Saved → outputs/06_absence_candidates.xlsx")
df_absence["status"].value_counts()

### b. Reading Back My Approvals

In `06_absence_candidates.xlsx`, type `Y` in `approve` for each candidate negative I accept (optionally a `confidence`), and save it as **`outputs/06_absence_APPROVED.xlsx`**.

In [ ]:
path = os.path.join(OUT, "06_absence_APPROVED.xlsx")
if os.path.exists(path):
    df_approved = pd.read_excel(path, dtype=str, keep_default_na=False)
    df_approved = df_approved[(df_approved["status"] == "Candidate negative") & (df_approved["approve"].str.strip().str.upper() == "Y")]
else:
    df_approved = df_absence.iloc[0:0]
    print("No approvals file yet: no negatives will be included in Part 7.")
print(f"{len(df_approved)} negatives approved")

## Part 7: Building the Attestation Rows to Paste

This part turns my verdicts and approvals into finished rows for the `attestations` sheet. The rows are saved as Excel and CSV, and copied to the clipboard tab-separated, because Excel spreads tab-separated text across columns when pasted (comma-separated text lands in one cell).

### a. Reading the Workbook's Current State

In [ ]:
#Save the workbook in Excel before running this: it reads the saved file
wb = openpyxl.load_workbook(WORKBOOK, read_only=True)
sheet = wb["attestations"]
HEADER = [c for c in next(sheet.iter_rows(min_row=1, max_row=1, max_col=16, values_only=True))]
last_row, existing_ids, existing_notes = 1, [], []
for i, r in enumerate(sheet.iter_rows(min_row=2, max_col=16, values_only=True), start=2):
    if r[0]:
        last_row = i
        existing_ids.append(int(re.sub(r"\D", "", str(r[0]))))
        existing_notes.append(str(r[15] or ""))
sources = {r[0]: {"rms_error_m": r[11]} for r in wb["sources"].iter_rows(min_row=2, values_only=True) if r[0]}
wb.close()
existing_markers = " ".join(existing_notes)
print(f"Last attestation row: {last_row}, next id: A-{max(existing_ids) + 1:04d}")

### b. Building the Rows

In [ ]:
#Date precision -> when_start, when_end, temporal_confidence
def date_fields(date):
    first, last = year_bounds(date, date)
    if first is None:
        return None, None, None
    if re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", str(date).strip()):
        return first, last, "High"
    return first, last, "Medium" if first == last else "Low"


new_rows = []
accepted = df_verdicts[df_verdicts["verdict"].isin(VERDICT_TO_TYPE)].copy()

#Text hits: earliest and latest per source, place and type (every erasure event kept)
text_hits = accepted[accepted["verdict"] != "geometry"].copy()
text_hits["attestation_type"] = text_hits["verdict"].map(VERDICT_TO_TYPE)
for (source_id, place_id, atype), g in text_hits.groupby(["source_id", "place_id", "attestation_type"]):
    g = g.sort_values(["date", "item_id", "page"])
    chosen = g if atype == "Erasure event" or len(g) == 1 else g.iloc[[0, -1]]
    for _, h in chosen.iterrows():
        ws, we, tconf = date_fields(h["date"])
        span = f" (earliest/latest of {len(g)} confirmed hits)" if len(chosen) == 2 else ""
        note = (f"KWIC: '...{h['left'][-120:]} {h['hit']} {h['right'][:120]}...' "
                f"[{h['match_type']} OCR match on '{h['phrase']}']{span} {h['verdict_note']}".strip())
        new_rows.append([place_id, source_id, atype, h["matched_text"], h["lang"] or "en", None, None, "None", None,
                         ws, we, tconf, h["confidence"] or h["suggested_confidence"], h["page_ref"],
                         f"{note} [OCR {h['hit_id']}]"])

#Map labels: merge reads under MAP_MERGE_M apart
map_hits = accepted[(accepted["verdict"] == "geometry") & (accepted["lat"] != "")]
for (source_id, place_id), g in map_hits.groupby(["source_id", "place_id"]):
    clusters = []
    for _, h in g.iterrows():
        point = (float(h["lat"]), float(h["lng"]))
        for c in clusters:
            if metres_between(point, c["centre"]) < MAP_MERGE_M:
                c["hits"].append(h)
                n = len(c["hits"])
                c["centre"] = ((c["centre"][0] * (n - 1) + point[0]) / n, (c["centre"][1] * (n - 1) + point[1]) / n)
                break
        else:
            clusters.append({"centre": point, "hits": [h]})
    rms = sources.get(source_id, {}).get("rms_error_m")
    rms = float(rms) if isinstance(rms, (int, float)) else 0.0
    for c in clusters:
        half = max(float(h["label_halfdiag_m"] or 0) for h in c["hits"])
        uncertainty = round((rms + half + LABEL_OFFSET_M) / 10) * 10
        first = c["hits"][0]
        ws, we, tconf = date_fields(first["date"])
        labels = "; ".join(sorted({h["matched_text"] for h in c["hits"]}))
        markers = " ".join(f"[OCR {h['hit_id']}]" for h in c["hits"])
        note = (f"OCR label(s) '{labels}' read on the georeferenced sheet; position = label centre (uncertainty = RMS "
                f"{rms:.0f} m + label half-size {half:.0f} m + offset {LABEL_OFFSET_M} m). {markers}")
        new_rows.append([place_id, source_id, "Geometry (Setting)", labels, first["lang"] or "en",
                         round(c["centre"][0], 6), round(c["centre"][1], 6), "Georeferenced map read", uncertainty,
                         ws, we, tconf, first["confidence"] or first["suggested_confidence"], first["page_ref"], note])

#Approved negatives
for _, r in df_approved.iterrows():
    group = manifest[manifest["source_id"] == r["source_id"]]
    years = [y for d in group["date"] for y in year_bounds(d, d) if y]
    new_rows.append([r["place_id"], r["source_id"], "Negative (searched-absent)", f"{r['place_name']} (searched)", "en",
                     None, None, "None", None, min(years) if years else None, max(years) if years else None,
                     "Medium" if years else None, r["confidence"] or "Medium",
                     "; ".join(group["citation"].replace("", "?"))[:250],
                     f"{r['note']} [OCR-NEG {r['source_id']} {r['place_id']}]"])
print(f"{len(new_rows)} rows built")

### c. Numbering, Exporting and Copying the Rows

In [ ]:
paste_rows, skipped = [], 0
for values in new_rows:
    marker = re.search(r"\[OCR(?:-NEG)? [^\]]+\]", values[14]).group(0)
    #Already in the workbook: don't paste it twice
    if marker in existing_markers:
        skipped += 1
        continue
    r = last_row + 1 + len(paste_rows)
    row = [f"A-{max(existing_ids) + 1 + len(paste_rows):04d}"] + values + [
        f'=IF($C{r}="","",IFERROR(VLOOKUP($C{r},sources!$A$2:$N$200,2,FALSE()),"?"))',
        f'=IF($Q{r}="","",IF($Q{r}="Map","Map",IF(OR($Q{r}="Photograph",$Q{r}="Postcard"),"Image",'
        f'IF(OR($Q{r}="Census or roll",$Q{r}="Report",$Q{r}="Newspaper",$Q{r}="Letter",$Q{r}="Memoir",'
        f'$Q{r}="Administrative",$Q{r}="Web compilation"),"Textual",IF($Q{r}="Legal claim","Legal/admin",'
        f'IF($Q{r}="Secondary","Secondary","Other"))))))',
        f'=IF($C{r}="","",IFERROR(VLOOKUP($C{r},sources!$A$2:$N$200,9,FALSE()),"?"))',
    ]
    #A tab or line break inside a value would push the rest of the row out of line
    paste_rows.append(["" if v is None else re.sub(r"[\t\r\n]+", " ", str(v)) for v in row])

df_paste = pd.DataFrame(paste_rows, columns=HEADER + ["medium (auto)", "medium_group (auto)", "source_origin (auto)"])
save_checkpoint(df_paste, "07_rows_to_paste", excel=True)

if paste_rows:
    #Copies the rows to the clipboard, tab-separated and without headers, ready for Cmd+V in Excel
    block = "\n".join("\t".join(r) for r in paste_rows)
    subprocess.run(["pbcopy"], input=block.encode("utf-8"), check=True)
    print(f"\n{len(paste_rows)} row(s) copied to the clipboard ({skipped} already in the sheet, left out).")
    print(f"PASTE AT: attestations sheet, cell A{last_row + 1}  (click it, then Cmd+V)")
else:
    print(f"Nothing new to paste ({skipped} already in the sheet).")
df_paste.head()

### d. Exporting to the Map

After pasting, save the workbook in Excel (Excel calculates the pasted formulas itself), then run the export. Read the DATA HEALTH report it prints before committing and publishing.

In [ ]:
#export.py uses paths relative to the project folder, so it is run from there
result = subprocess.run(["python3", "export.py"], cwd=PROJECT, capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-2000:])